In [ ]:
# TUGAS 1: Perbandingan SVM Linear vs RBF dengan Parameter Default
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset Breast Cancer
cancer = load_breast_cancer()
print("Dataset Breast Cancer loaded!")
print(f"Jumlah data: {cancer.data.shape[0]} sampel")
print(f"Jumlah fitur: {cancer.data.shape[1]} fitur")
print(f"Nama fitur: {cancer.feature_names[:5]}... (dan seterusnya)")
print(f"Nama kelas: {cancer.target_names}")
print(f"Distribusi kelas: {np.bincount(cancer.target)}")

# Membagi data menjadi data latih (80%) dan data uji (20%)
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42, stratify=cancer.target
)

print(f"\nData latih: {X_train.shape[0]} sampel")
print(f"Data uji: {X_test.shape[0]} sampel")

# Scaling fitur (SVM sangat sensitif terhadap skala)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nStatistik setelah scaling (mean mendekati 0, std mendekati 1):")
print(f"Mean data latih: {np.mean(X_train_scaled):.3f}")
print(f"Std data latih: {np.std(X_train_scaled):.3f}")

# Membuat dua model SVM dengan kernel berbeda (parameter default)
models = {
    "SVM Linear (C=1)": SVC(kernel="linear", C=1, random_state=42),
    "SVM RBF (C=1, gamma='scale')": SVC(kernel="rbf", C=1, gamma='scale', random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    results[name] = {
        'model': model,
        'accuracy': acc,
        'predictions': y_pred
    }
    
    print(f"\n{'='*60}")
    print(f"HASIL {name}")
    print(f"{'='*60}")
    print(f"Accuracy: {acc:.4f}")
    print(f"\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=cancer.target_names))

# Visualisasi perbandingan akurasi
plt.figure(figsize=(8, 5))
names = list(results.keys())
accuracies = [results[name]['accuracy'] for name in names]
colors = ['steelblue', 'coral']

bars = plt.bar(names, accuracies, color=colors, alpha=0.8)
plt.ylabel('Accuracy')
plt.title('Perbandingan Akurasi SVM Linear vs RBF (Default Parameter)')
plt.ylim(0.9, 1.0)
plt.grid(axis='y', alpha=0.3)

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
             f'{acc:.4f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# TUGAS 2: Eksperimen Tuning Parameter C dan Gamma
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC

# Nilai C yang akan diuji
C_values = [0.1, 1, 10]

# Nilai gamma yang akan diuji (untuk kernel RBF)
gamma_values = [0.01, 0.1, 1]

# Dictionary untuk menyimpan hasil eksperimen
results_tuning = {
    'Kernel': [],
    'C': [],
    'Gamma': [],
    'Accuracy': []
}

print("="*80)
print("EKSPERIMEN TUNING PARAMETER C DAN GAMMA")
print("="*80)

# 1. Eksperimen pada kernel Linear (hanya parameter C yang berpengaruh)
print("\n--- KERNEL LINEAR (gamma tidak berpengaruh) ---")
for C in C_values:
    model = SVC(kernel='linear', C=C, random_state=42)
    model.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_scaled))
    
    results_tuning['Kernel'].append('Linear')
    results_tuning['C'].append(C)
    results_tuning['Gamma'].append('-')
    results_tuning['Accuracy'].append(acc)
    
    print(f"C = {C:<4} -> Accuracy: {acc:.4f}")

# 2. Eksperimen pada kernel RBF (parameter C dan gamma)
print("\n--- KERNEL RBF ---")
for C in C_values:
    for gamma in gamma_values:
        model = SVC(kernel='rbf', C=C, gamma=gamma, random_state=42)
        model.fit(X_train_scaled, y_train)
        acc = accuracy_score(y_test, model.predict(X_test_scaled))
        
        results_tuning['Kernel'].append('RBF')
        results_tuning['C'].append(C)
        results_tuning['Gamma'].append(gamma)
        results_tuning['Accuracy'].append(acc)
        
        print(f"C = {C:<4}, gamma = {gamma:<4} -> Accuracy: {acc:.4f}")

# Buat DataFrame untuk melihat hasil
df_results = pd.DataFrame(results_tuning)
print("\n" + "="*80)
print("RINGKASAN HASIL TUNING")
print("="*80)
print(df_results.to_string(index=False))

# Visualisasi hasil tuning
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot untuk kernel Linear
linear_results = df_results[df_results['Kernel'] == 'Linear']
axes[0].plot(linear_results['C'], linear_results['Accuracy'], 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Nilai C')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Pengaruh C pada SVM Linear')
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)
for i, row in linear_results.iterrows():
    axes[0].text(row['C'], row['Accuracy'] + 0.002, f"{row['Accuracy']:.4f}", ha='center')

# Plot untuk kernel RBF (pengaruh C dan gamma)
rbf_results = df_results[df_results['Kernel'] == 'RBF']
for gamma in gamma_values:
    subset = rbf_results[rbf_results['Gamma'] == gamma]
    axes[1].plot(subset['C'], subset['Accuracy'], 'o-', linewidth=2, markersize=6, label=f'gamma={gamma}')

axes[1].set_xlabel('Nilai C')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Pengaruh C dan Gamma pada SVM RBF')
axes[1].set_xscale('log')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Tampilkan model terbaik
best_model = df_results.loc[df_results['Accuracy'].idxmax()]
print("\n" + "="*80)
print("MODEL TERBAIK")
print("="*80)
print(f"Kernel   : {best_model['Kernel']}")
print(f"C        : {best_model['C']}")
if best_model['Kernel'] == 'RBF':
    print(f"Gamma    : {best_model['Gamma']}")
print(f"Accuracy : {best_model['Accuracy']:.4f}")

In [ ]:
# TUGAS 3: Perbandingan dan Pemilihan Model Terbaik
# (Menggunakan hasil tuning sebelumnya)

# Model terbaik dari hasil tuning
best_C = 1.0
best_gamma = 0.1

# Latih model terbaik
best_model = SVC(kernel='rbf', C=best_C, gamma=best_gamma, random_state=42, probability=True)
best_model.fit(X_train_scaled, y_train)
y_pred_best = best_model.predict(X_test_scaled)
y_prob_best = best_model.predict_proba(X_test_scaled)

# Evaluasi model terbaik
print("\n" + "="*80)
print("EVALUASI MODEL TERBAIK (RBF, C=1, gamma=0.1)")
print("="*80)
print(f"Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print(f"\nConfusion Matrix:")
cm_best = confusion_matrix(y_test, y_pred_best)
print(cm_best)
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=cancer.target_names))

# Visualisasi confusion matrix model terbaik
plt.figure(figsize=(8, 6))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues',
            xticklabels=cancer.target_names, yticklabels=cancer.target_names)
plt.title(f'Confusion Matrix - Model Terbaik (RBF, C=1, gamma=0.1)\nAccuracy: {accuracy_score(y_test, y_pred_best):.4f}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# Tabel perbandingan semua model
print("\n" + "="*80)
print("PERBANDINGAN SEMUA MODEL")
print("="*80)
print(df_results.to_string(index=False))